# Phase 1-4: Eyes-Open ↔ Eyes-Closed 전환 동역학 분석

**이 분석은 EDCC 모델의 핵심 가설을 검증합니다.**

**핵심 가설**: EO↔EC 전환 시의 동적 패턴이 class를 구분하는 핵심 특징이다.

**분석 내용**:
1. 전환 시점 전후 ±5초 구간의 시간-주파수 분석
2. Alpha reactivity 시간 곡선 (class별)
3. 전환 응답 시간 측정
4. Theta/Delta 전환 패턴
5. Photic Driving Response 분석

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as scipy_signal
from scipy import stats

sys.path.insert(0, os.path.abspath('..'))

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

DATASET_PATH = '../local/dataset/caueeg-dataset'
SAMPLING_RATE = 200
TRANSITION_WINDOW = 5 * SAMPLING_RATE  # ±5초 = 1000 samples 각 방향

FREQ_BANDS = {
    'Delta': (0.5, 4), 'Theta': (4, 8), 'Alpha': (8, 13), 
    'Beta': (13, 30), 'Gamma': (30, 45),
}

# Occipital 채널 (Alpha reactivity 분석용)
OCC_CHANNELS = [4, 9]  # O1, O2

In [ ]:
from datasets.caueeg_script import load_caueeg_task_datasets

task_config, train_ds, val_ds, test_ds = load_caueeg_task_datasets(
    DATASET_PATH, task='dementia', load_event=True, file_format='edf'
)
class_names = task_config['class_label_to_name']
colors = {'Normal': '#2ecc71', 'MCI': '#f39c12', 'Dementia': '#e74c3c'}

## 1. EO→EC 전환 시점 추출 및 시간-주파수 분석

각 녹음에서 EO→EC 전환 시점을 찾고, 전후 ±5초 구간의 Alpha power 시간 변화를 추적합니다.

In [ ]:
def find_transitions(events, signal_length, transition_type='EO_to_EC'):
    """EO→EC 또는 EC→EO 전환 시점을 찾습니다."""
    eo_ec_events = [(idx, name) for idx, name in events 
                     if name in ('Eyes Open', 'Eyes Closed')]
    
    transitions = []
    for i in range(len(eo_ec_events) - 1):
        curr_idx, curr_name = eo_ec_events[i]
        next_idx, next_name = eo_ec_events[i + 1]
        
        if transition_type == 'EO_to_EC' and curr_name == 'Eyes Open' and next_name == 'Eyes Closed':
            # 전환 시점은 Eyes Closed가 시작되는 지점
            if next_idx - TRANSITION_WINDOW >= 0 and next_idx + TRANSITION_WINDOW < signal_length:
                transitions.append(next_idx)
        elif transition_type == 'EC_to_EO' and curr_name == 'Eyes Closed' and next_name == 'Eyes Open':
            if next_idx - TRANSITION_WINDOW >= 0 and next_idx + TRANSITION_WINDOW < signal_length:
                transitions.append(next_idx)
    
    return transitions

def compute_sliding_band_power(sig, fs, band, window_sec=1.0, stride_sec=0.25):
    """슬라이딩 윈도우로 시간에 따른 band power를 계산"""
    window_samples = int(window_sec * fs)
    stride_samples = int(stride_sec * fs)
    
    powers = []
    times = []
    
    for start in range(0, len(sig) - window_samples + 1, stride_samples):
        segment = sig[start:start + window_samples]
        freqs, psd = scipy_signal.welch(segment, fs=fs, nperseg=min(window_samples, 200))
        idx = np.logical_and(freqs >= band[0], freqs <= band[1])
        power = np.trapz(psd[idx], freqs[idx])
        powers.append(power)
        times.append((start + window_samples / 2) / fs)
    
    return np.array(times), np.array(powers)

print("Utility functions defined.")

In [ ]:
# EO→EC 전환 전후의 Alpha power 시간 곡선 수집
print("Extracting EO→EC transition dynamics...")

# class별로 Alpha power 시간 곡선 수집
transition_curves = {cls: [] for cls in ['Normal', 'MCI', 'Dementia']}
transition_counts = {cls: 0 for cls in ['Normal', 'MCI', 'Dementia']}

for i in range(len(train_ds)):
    sample = train_ds[i]
    eeg = sample['signal'][:19]
    events = sample.get('event', [])
    cls = class_names[sample['class_label']]
    
    transitions = find_transitions(events, eeg.shape[1], 'EO_to_EC')
    
    for t_idx in transitions:
        # 전환 시점 전후 ±5초 구간
        start = t_idx - TRANSITION_WINDOW
        end = t_idx + TRANSITION_WINDOW
        
        # Occipital 채널 평균의 Alpha power 시간 곡선
        occ_signal = np.mean(eeg[OCC_CHANNELS, start:end], axis=0)
        
        # 슬라이딩 윈도우로 Alpha power 계산
        times, alpha_powers = compute_sliding_band_power(
            occ_signal, SAMPLING_RATE, (8, 13), window_sec=1.0, stride_sec=0.25
        )
        # 시간을 전환 시점 기준으로 조정
        times = times - 5.0  # 0 = 전환 시점
        
        transition_curves[cls].append((times, alpha_powers))
        transition_counts[cls] += 1
    
    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(train_ds)} processed")

print(f"\nEO→EC transitions found:")
for cls, count in transition_counts.items():
    print(f"  {cls}: {count} transitions")

## 2. EO→EC 전환 시 Alpha Power 시간 곡선 (Class별)

**예상**: 
- Normal: 눈 감으면 Alpha 급격 증가 (강한 Alpha reactivity)
- Dementia: Alpha reactivity 둔화 또는 소실
- MCI: 중간 패턴 또는 다른 양상?

In [ ]:
# Class별 평균 Alpha power 전환 곡선
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 개별 곡선 + 평균
ax = axes[0]
for cls in ['Normal', 'MCI', 'Dementia']:
    if not transition_curves[cls]:
        continue
    
    # 공통 시간 축에 보간
    common_times = np.linspace(-4.5, 4.5, 36)
    interpolated = []
    
    for times, powers in transition_curves[cls]:
        if len(times) >= 2:
            interp_powers = np.interp(common_times, times, powers)
            interpolated.append(interp_powers)
    
    if interpolated:
        arr = np.array(interpolated)
        mean_curve = np.mean(arr, axis=0)
        std_curve = np.std(arr, axis=0)
        sem_curve = std_curve / np.sqrt(len(arr))
        
        ax.plot(common_times, mean_curve, color=colors[cls], linewidth=2.5, 
                label=f'{cls} (n={len(arr)})')
        ax.fill_between(common_times, mean_curve - sem_curve, mean_curve + sem_curve,
                        color=colors[cls], alpha=0.2)

ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, label='EO→EC transition')
ax.set_xlabel('Time relative to transition (seconds)')
ax.set_ylabel('Occipital Alpha Power')
ax.set_title('EO→EC Transition: Alpha Power Time Course')
ax.legend()
ax.grid(True, alpha=0.3)

# 정규화된 곡선 (전환 전 baseline 대비)
ax = axes[1]
for cls in ['Normal', 'MCI', 'Dementia']:
    if not transition_curves[cls]:
        continue
    
    common_times = np.linspace(-4.5, 4.5, 36)
    interpolated = []
    
    for times, powers in transition_curves[cls]:
        if len(times) >= 2:
            interp_powers = np.interp(common_times, times, powers)
            # Baseline: 전환 전 3-5초 구간 평균
            baseline_idx = common_times < -2
            if np.any(baseline_idx):
                baseline = np.mean(interp_powers[baseline_idx])
                if baseline > 0:
                    interp_powers = interp_powers / baseline
                    interpolated.append(interp_powers)
    
    if interpolated:
        arr = np.array(interpolated)
        mean_curve = np.mean(arr, axis=0)
        sem_curve = np.std(arr, axis=0) / np.sqrt(len(arr))
        
        ax.plot(common_times, mean_curve, color=colors[cls], linewidth=2.5, 
                label=f'{cls} (n={len(arr)})')
        ax.fill_between(common_times, mean_curve - sem_curve, mean_curve + sem_curve,
                        color=colors[cls], alpha=0.2)

ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5)
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Time relative to transition (seconds)')
ax.set_ylabel('Normalized Alpha Power (baseline=1)')
ax.set_title('EO→EC Transition: Normalized Alpha Power')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 전환 응답 시간 및 Alpha 변화량 측정

In [ ]:
# 전환 응답 측정: 전환 후 Alpha가 최대값에 도달하는 시간, 변화량
response_records = []

for cls in ['Normal', 'MCI', 'Dementia']:
    common_times = np.linspace(-4.5, 4.5, 36)
    
    for times, powers in transition_curves[cls]:
        if len(times) < 2:
            continue
        interp_powers = np.interp(common_times, times, powers)
        
        # Baseline: 전환 전 구간 (-4.5 ~ -1초) 평균
        pre_idx = (common_times >= -4.5) & (common_times <= -1.0)
        baseline_power = np.mean(interp_powers[pre_idx])
        
        # Post-transition: 전환 후 구간 (0 ~ 4.5초)
        post_idx = common_times >= 0
        post_powers = interp_powers[post_idx]
        post_times = common_times[post_idx]
        
        if len(post_powers) > 0 and baseline_power > 0:
            # Peak alpha power와 도달 시간
            peak_idx = np.argmax(post_powers)
            peak_power = post_powers[peak_idx]
            peak_time = post_times[peak_idx]
            
            # Alpha 변화량 (absolute & relative)
            alpha_change = peak_power - baseline_power
            alpha_change_rel = alpha_change / baseline_power
            
            response_records.append({
                'class_name': cls,
                'baseline_power': baseline_power,
                'peak_power': peak_power,
                'peak_time_sec': peak_time,
                'alpha_change': alpha_change,
                'alpha_change_rel': alpha_change_rel,
            })

df_response = pd.DataFrame(response_records)

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Alpha 변화량 (상대값)
sns.boxplot(data=df_response, x='class_name', y='alpha_change_rel',
            order=['Normal', 'MCI', 'Dementia'], palette=colors, ax=axes[0])
axes[0].axhline(y=0, color='gray', linestyle='--')
axes[0].set_title('Alpha Change (Relative to Baseline)')
axes[0].set_ylabel('(Peak - Baseline) / Baseline')

# Peak 도달 시간
sns.boxplot(data=df_response, x='class_name', y='peak_time_sec',
            order=['Normal', 'MCI', 'Dementia'], palette=colors, ax=axes[1])
axes[1].set_title('Time to Peak Alpha After Transition')
axes[1].set_ylabel('Time (seconds)')

# Baseline alpha power
sns.boxplot(data=df_response, x='class_name', y='baseline_power',
            order=['Normal', 'MCI', 'Dementia'], palette=colors, ax=axes[2])
axes[2].set_title('Baseline Alpha Power (Pre-transition)')
axes[2].set_ylabel('Alpha Power')

plt.tight_layout()
plt.show()

# 통계
print("=== 전환 응답 통계 ===")
for metric in ['alpha_change_rel', 'peak_time_sec', 'baseline_power']:
    print(f"\n{metric}:")
    for cls in ['Normal', 'MCI', 'Dementia']:
        vals = df_response[df_response['class_name'] == cls][metric]
        print(f"  {cls}: mean={vals.mean():.3f}, std={vals.std():.3f}")
    
    # Kruskal-Wallis
    groups = [df_response[df_response['class_name'] == c][metric] for c in ['Normal', 'MCI', 'Dementia']]
    if all(len(g) > 0 for g in groups):
        stat, p = stats.kruskal(*groups)
        print(f"  Kruskal-Wallis: H={stat:.3f}, p={p:.2e}")

## 4. 다중 대역 전환 동역학

전환 시점 전후의 Delta, Theta, Alpha, Beta 파워 변화를 동시에 추적합니다.

In [ ]:
# 다중 대역 전환 곡선 수집 (최대 300개 녹음 샘플링하여 속도 최적화)
print("Computing multi-band transition dynamics (sampled)...")
multi_band_curves = {cls: {band: [] for band in FREQ_BANDS} for cls in ['Normal', 'MCI', 'Dementia']}

MAX_SAMPLES = 300
np.random.seed(42)
sample_indices = np.random.choice(len(train_ds), min(MAX_SAMPLES, len(train_ds)), replace=False)

for count, i in enumerate(sample_indices):
    sample = train_ds[i]
    eeg = sample['signal'][:19]
    events = sample.get('event', [])
    cls = class_names[sample['class_label']]
    
    transitions = find_transitions(events, eeg.shape[1], 'EO_to_EC')
    
    for t_idx in transitions[:3]:  # 녹음당 최대 3개 전환만
        start = t_idx - TRANSITION_WINDOW
        end = t_idx + TRANSITION_WINDOW
        occ_signal = np.mean(eeg[OCC_CHANNELS, start:end], axis=0)
        
        for band_name, band_range in FREQ_BANDS.items():
            times, powers = compute_sliding_band_power(
                occ_signal, SAMPLING_RATE, band_range, window_sec=1.0, stride_sec=0.5
            )
            times = times - 5.0
            multi_band_curves[cls][band_name].append((times, powers))
    
    if (count + 1) % 100 == 0:
        print(f"  {count+1}/{len(sample_indices)} processed")

print("Done.")
for cls in ['Normal', 'MCI', 'Dementia']:
    print(f"  {cls}: {len(multi_band_curves[cls]['Alpha'])} transition curves")

# 시각화: Class별 다중 대역 전환 곡선
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
band_colors = {'Delta': '#1f77b4', 'Theta': '#ff7f0e', 'Alpha': '#2ca02c', 
               'Beta': '#d62728', 'Gamma': '#9467bd'}
common_times = np.linspace(-4.5, 4.5, 18)

for ax_idx, cls in enumerate(['Normal', 'MCI', 'Dementia']):
    ax = axes[ax_idx]
    
    for band_name in ['Delta', 'Theta', 'Alpha', 'Beta']:
        curves = multi_band_curves[cls][band_name]
        if not curves:
            continue
        
        interpolated = []
        for times, powers in curves:
            if len(times) >= 2:
                interp = np.interp(common_times, times, powers)
                baseline = np.mean(interp[common_times < -2])
                if baseline > 0:
                    interpolated.append(interp / baseline)
        
        if interpolated:
            arr = np.array(interpolated)
            mean_curve = np.mean(arr, axis=0)
            sem = np.std(arr, axis=0) / np.sqrt(len(arr))
            
            ax.plot(common_times, mean_curve, color=band_colors[band_name], 
                    linewidth=2, label=band_name)
            ax.fill_between(common_times, mean_curve - sem, mean_curve + sem,
                           color=band_colors[band_name], alpha=0.15)
    
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5)
    ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(f'{cls}', fontweight='bold')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Normalized Power' if ax_idx == 0 else '')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('EO→EC Transition: Multi-band Power Dynamics (Occipital)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 요약 및 EDCC 설계 시사점

**핵심 결과 체크리스트**:
- [ ] Alpha reactivity가 Normal > MCI > Dementia 순서로 감소하는지
- [ ] 전환 응답 시간이 class 간에 차이가 있는지
- [ ] Theta/Delta 전환 반응이 class별로 다른 패턴을 보이는지
- [ ] 전환 구간의 동적 패턴이 정적 구간 분석보다 더 많은 정보를 제공하는지

**EDCC 설계에 대한 시사점**:
- Alpha reactivity 차이가 유의미하면 → Event-Conditioned Mamba의 전환 구간 특별 처리 정당화
- 응답 시간 차이가 있으면 → Core token trajectory에서 시간적 패턴이 중요
- 다중 대역 동시 변화 패턴 → Multi-scale Tokenization의 필요성 뒷받침

**다음 단계**: Phase 1-5에서 채널 간 연결성을 분석하여 Region-Level GCN 설계 근거를 마련합니다.